In [1]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

import xgboost as xgb
import joblib


In [2]:
df = pd.read_csv("../data/raw/flights_sample.csv")
print(df.head())
print(df.info())
print("Missing values:\n", df.isnull().sum())


    airline source destination departure_date  stops  duration_mins  \
0    Indigo    HYD         DEL     2025-10-15      0            120   
1  AirIndia    HYD         BOM     2025-10-20      1            150   
2  SpiceJet    BOM         GOI     2025-11-05      0             90   
3   Vistara    DEL         HYD     2025-12-01      0            135   

   days_to_dep  base_price  
0           40        4500  
1           45        4200  
2           10        3200  
3           25        5000  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   airline         4 non-null      object
 1   source          4 non-null      object
 2   destination     4 non-null      object
 3   departure_date  4 non-null      object
 4   stops           4 non-null      int64 
 5   duration_mins   4 non-null      int64 
 6   days_to_dep     4 non-null      int64 
 7 

In [3]:
# convert departure_date if you want date features (optional)
# df['departure_date'] = pd.to_datetime(df['departure_date'])
# df['days_to_departure'] = (df['departure_date'] - pd.Timestamp.today()).dt.days

# One-hot encode categorical columns used in your example
categorical_cols = ['airline','source','destination']
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Drop target and any raw date column referenced earlier
X = df.drop(columns=['base_price','departure_date'], errors='ignore')
y = df['base_price']

print("Feature count:", X.shape[1])
print("Sample features:", X.columns.tolist()[:20])


Feature count: 11
Sample features: ['stops', 'duration_mins', 'days_to_dep', 'airline_Indigo', 'airline_SpiceJet', 'airline_Vistara', 'source_DEL', 'source_HYD', 'destination_DEL', 'destination_GOI', 'destination_HYD']


In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [5]:
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
print("RF MAE:", mean_absolute_error(y_test, y_pred))


RF MAE: 246.0


In [6]:
xgb_model = xgb.XGBRegressor(n_estimators=200, learning_rate=0.1, random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)
print("XGB MAE:", mean_absolute_error(y_test, y_pred_xgb))


XGB MAE: 799.97216796875


In [7]:
os.makedirs("../data/artifacts", exist_ok=True)
artifact_path = "../data/artifacts/flight_price_model.pkl"

artifact = {
    "model": xgb_model,                    # or rf if RF was better
    "feature_columns": X.columns.tolist()  # save the exact feature order
}
joblib.dump(artifact, artifact_path)
print("Saved model + columns to:", artifact_path)


Saved model + columns to: ../data/artifacts/flight_price_model.pkl


In [8]:
import os, joblib
os.makedirs("../data/artifacts", exist_ok=True)

artifact = {
    "model": xgb_model,                       # or rf if RF was best
    "feature_columns": X.columns.tolist()     # X is your final feature dataframe used during training
}
joblib.dump(artifact, "../data/artifacts/flight_price_model_clean.pkl")
print("Saved to ../data/artifacts/flight_price_model_clean.pkl")


Saved to ../data/artifacts/flight_price_model_clean.pkl
